[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NASA-Gradient-Descent-Farmers/NASA-warm-up/blob/sant-calibration/Notebooks/yield_panel_sources.ipynb)

# Which NASA sources help? IMERG, CHIRPS, MODIS NDVI, SMAP and IGBP on the yield panel

* **Author:** Santiago (Gradient Descent Farmers) · **Branch:** `sant-calibration`
* **Follows:** `yield_panel.ipynb`, where POWER weather barely beat "municipality average + trend", although 35-55% of what the bar leaves unexplained is shared by neighbouring municipalities in the same year.
* **Code:** `src/yield_panel/gridded.py` (downloads and sampling), `enrich.py` (joins), `ablation.py` (evaluation).

Each source is pulled as a box over central Mexico with server-side subsetting where available, then averaged around each municipality point:

| Source | What | Grid | Years | Around each municipality |
|---|---|---|---|---|
| GPM IMERG (Final; Late Run after 2025-09-30) | daily rain | 0.1° | 2002- | 15 km mean |
| CHIRPS v2.0 | daily rain | 0.05° | 2002- | 10 km mean |
| MODIS MOD13C2 | monthly NDVI, EVI | 0.05° | 2003- | 15 km, weighted by MCD12C1 cropland share |
| SMAP L4 | root-zone and surface soil moisture, one snapshot a day | 9 km | 2015- | 10 km mean |
| IGBP-DIS | field capacity, wilting point, available water, soil C and N | ~9 km | static | point |

**NDVI is different from the others.** It measures how green the crop got, which is partly the outcome itself. "Weather + NDVI" fits monitoring during the season; for replaying weather scenarios use the weather-only sets.

## 0. Setup and Earthdata login

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/NASA-Gradient-Descent-Farmers/NASA-warm-up.git"
BRANCH = "sant-calibration"

if "google.colab" in sys.modules:
    if not Path("/content/NASA-warm-up").exists():
        !git clone -q --branch {BRANCH} {REPO_URL} /content/NASA-warm-up
    !pip install -q "earthaccess>=0.11" pyhdf
    ROOT = Path("/content/NASA-warm-up")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "yield_panel").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

# Earthdata: paste a token (urs.earthdata.nasa.gov -> Generate Token) when prompted. Never commit it.
if not os.environ.get("EARTHDATA_TOKEN"):
    from getpass import getpass
    os.environ["EARTHDATA_TOKEN"] = getpass("Earthdata token: ")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.yield_panel import ablation, build, enrich, models as pm

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
PANEL = Path("data/yield_panel/panel_central.csv")
ENRICHED = Path("data/yield_panel/panel_central_enriched.csv")

## 1. Build and enrich (cached)

First run, central Mexico: panel ~3 min; downloads ~30 min in total (IMERG ~8,400 daily boxes, CHIRPS ~1 min per year with HTTP range reads, MODIS ~280 monthly boxes, SMAP ~3,900 daily boxes, MCD12C1 one 1.2 GB file that is cut to the box and deleted); joining ~4 min.

In [ ]:
if not PANEL.exists():
    panel, info = build.build_panel(range(2003, 2026), states=build.CENTRAL_STATES)
    PANEL.parent.mkdir(parents=True, exist_ok=True)
    panel.to_csv(PANEL, index=False)
if not ENRICHED.exists():
    enrich.enrich(pd.read_csv(PANEL)).to_csv(ENRICHED, index=False)
data = pd.read_csv(ENRICHED)
print(data.shape)

### Do the rain sources agree?

If POWER, IMERG and CHIRPS told the same story, swapping them could not change the results.

In [ ]:
rain = data.drop_duplicates(["state_id", "mun_id", "year"])[["state_id", "mun_id", "rain_season_mm",
                                                             "imerg_rain_season_mm", "chirps_rain_season_mm"]]
cols = ["rain_season_mm", "imerg_rain_season_mm", "chirps_rain_season_mm"]
within = rain[cols] - rain.groupby(["state_id", "mun_id"])[cols].transform("mean")
print("May-Oct rain, mean mm:", rain[cols].mean().round(0).to_dict())
print("Year-to-year correlation within municipalities:")
within.corr().round(2)

## 2. Ablation: every feature set against the same bar

Same design as before: expanding window, each test year 2014-2025 predicted from earlier years only, every feature set scored on the same rows, bootstrap over whole test years. `fe` = fixed-effects ridge, `gbm` = boosted residual. Skill = 1 − RMSE / RMSE(average + trend).

Running it takes ~30 minutes; it saves CSVs under `data/yield_panel/ablation/`, so later runs just load them.

In [ ]:
OUT = Path("data/yield_panel/ablation")
if not (OUT / "feature_sets.csv").exists():
    !python -m src.yield_panel.ablation --panel {ENRICHED}
results = pd.read_csv(OUT / "feature_sets.csv")
results["significant"] = results["gain_ci_low"] > 0
best = results.loc[results.groupby(["crop", "target", "features"])["skill"].idxmax()]
table = best.pivot_table(index="features", columns=["crop", "target"], values="skill")
order = list(ablation.feature_sets(data.columns))
table.reindex(order).round(3)

In [ ]:
sig = results[results.significant].sort_values("skill", ascending=False)
print("Feature sets whose RMSE gain over the bar has a 95% interval above zero:")
sig[["crop", "target", "features", "model", "skill", "gain_ci_low", "gain_ci_high"]].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(table.reindex(order).to_numpy(), cmap="RdBu", vmin=-0.15, vmax=0.15, aspect="auto")
ax.set_yticks(range(len(order)), order)
ax.set_xticks(range(table.shape[1]), [f"{c}\n{t.replace('_t_ha', '')}" for c, t in table.columns])
for (i, j), v in np.ndenumerate(table.reindex(order).to_numpy()):
    ax.text(j, i, f"{v:+.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="skill vs average + trend (best of fe/gbm)")
ax.set_title("Which sources help? (blue = beats the bar)")
fig.tight_layout()
plt.show()

## 3. SMAP (2015 onwards only)

SMAP starts in April 2015, so it gets its own comparison: trained from 2015, tested 2020-2025.

In [ ]:
pd.read_csv(OUT / "smap.csv")[["crop", "target", "features", "model", "skill", "gain_ci_low", "gain_ci_high", "rows"]].round(3)

## 4. What is the shared shock: weather, or the way SIAP reports?

Leave-one-out share of the bar's residual that a municipality shares with others in the same group and year. SIAP compiles its statistics by district (DDR). If neighbours move together because the same office estimated them, the share should be higher within a DDR than within a weather cell of similar size.

In [ ]:
pd.read_csv(OUT / "shared_variance.csv").round(3)

District areas are compact too, so this alone could still be geography. The direct test holds the weather fixed: take pairs of municipalities in the **same POWER cell and year**, and compare pairs in the same district with pairs in different districts. Weather cannot explain a difference between those two groups. The last column counts non-zero values that are *exactly* identical to another municipality's in the same district and year.

In [ ]:
pd.read_csv(OUT / "district_check.csv").round(3)

### The ablation again, at SIAP's own unit

If much of the municipal variation is the district office's estimate spread over its municipalities, the district is the real unit of observation. Here municipalities are aggregated to districts (summed areas and production, sown-area-weighted features) and the same evaluation is rerun.

In [ ]:
district = pd.read_csv(OUT / "district_level.csv")
district["significant"] = district["gain_ci_low"] > 0
best_d = district.loc[district.groupby(["crop", "target", "features"])["skill"].idxmax()]
best_d.pivot_table(index="features", columns=["crop", "target"], values="skill").reindex(ablation.DISTRICT_SETS).round(3)

In [ ]:
district[district.significant][["crop", "target", "features", "model", "skill", "gain_ci_low", "gain_ci_high", "rows"]].round(3)

## 5. What the NDVI model learned (maize failure)

In [ ]:
crop, target = "maize", "failure_rate"
weather, _ = ablation.feature_sets(data.columns)["power + ndvi"]
d, feats = pm.add_terms(data[data.crop == crop].dropna(subset=[target, *weather]), weather)
fit = pm.FixedEffects(["year", *feats], alpha=float(len(feats))).fit(d, target)
within_sd = (d[feats] - d.groupby(pm.UNIT)[feats].transform("mean")).std()
effects = (fit.coef_.drop("year") * within_sd).rename("effect_per_sd")
effects.reindex(effects.abs().sort_values(ascending=False).index).head(12).round(4)

## 6. Reading the results

* A source "helps" only if its gain interval stays above zero. Point estimates of a few percent with intervals around zero are noise at 12 test years.
* If better rain does not help, the bottleneck is not rain measurement. The district check says much of the shared variation follows SIAP's district boundaries, not the weather.
* Municipal SIAP numbers are partly district estimates spread over municipalities. For modelling, the district is the honest unit; for a farmer-facing tool, results should be presented at district or regional level.
* NDVI's gain on failure is real but partly circular (a failed field is less green). It supports an **in-season early warning**, not a pre-season choice.